# Bank of Canada: policy rate, CORRA, yields and the Canadian dollar

Source: Bank of Canada Valet API, series V39079, AVG.INTWO, BD.CDN.2YR/5YR/10YR/LONG.DQ.YLD and FXUSDCAD (business daily). The last table also uses Statistics Canada table 14-10-0287-01 through `analysis_ready/unemployment_rate_by_age_monthly.csv`, so run notebook 01 first. Run all cells; charts go to `outputs/` and the tables behind them to `analysis_ready/`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
pd.options.display.max_colwidth = 90
pd.options.display.float_format = "{:,.2f}".format


def find_hub_root():
    """Walk up from this notebook's folder until we find the hub (it contains processed/boc)."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "processed" / "boc").exists():
            return folder
    raise FileNotFoundError(
        "Could not find the hub folder (a folder that contains processed/boc). "
        "Open this notebook from inside the repository and run scripts/acquisition/boc/download_boc_series.py first."
    )


HUB = find_hub_root()
OUTPUTS = HUB / "outputs"                # charts are saved here
ANALYSIS_READY = HUB / "analysis_ready"  # summary tables are saved here
OUTPUTS.mkdir(exist_ok=True)
ANALYSIS_READY.mkdir(exist_ok=True)


# Chart style: light surface, thin lines, quiet grid. Colours are used in a fixed order
# (blue, orange, aqua, ...) so a series keeps the same colour in every chart. Gray is for context.
SURFACE, INK, INK2 = "#fcfcfb", "#0b0b0b", "#52514e"
GRID, AXIS, MUTED = "#e1e0d9", "#c3c2b7", "#898781"
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, RED = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#e34948"
)
SERIES_COLOURS = [BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "Helvetica Neue", "Arial", "DejaVu Sans"],
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "figure.dpi": 110,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.titlecolor": INK, "legend.frameon": False,
})


def subtitle(ax, text):
    """Small grey line under the chart title (we use it to show exactly which series is plotted)."""
    ax.text(0, 1.02, text, transform=ax.transAxes, fontsize=9.5, color=INK2, va="bottom", ha="left")


def spread(values, min_gap):
    """Nudge label positions apart so direct labels do not overlap."""
    values = np.asarray(values, dtype=float)
    order = np.argsort(values)
    pos = values[order].copy()
    for i in range(1, len(pos)):
        if pos[i] - pos[i - 1] < min_gap:
            pos[i] = pos[i - 1] + min_gap
    out = np.empty_like(pos)
    out[order] = pos
    return out


def direct_labels(ax, items):
    """Label each line at its right-hand end. items = [(text, series), ...]"""
    low, high = ax.get_ylim()
    ys = spread([s.iloc[-1] for _, s in items], 0.06 * (high - low))
    for (text, s), y in zip(items, ys):
        ax.annotate(text, (s.index[-1], y), xytext=(6, 0), textcoords="offset points",
                    va="center", fontsize=9, color=INK2)


def save_fig(fig, name):
    path = OUTPUTS / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("Saved chart:", path.relative_to(HUB))


def load_dataset(dataset_id):
    """Read a processed file by its dataset ID, e.g. 'boc_v39079_policy_rate_daily'."""
    source = dataset_id.split("_", 1)[0]
    path = HUB / "processed" / source / dataset_id / f"{dataset_id}.csv"
    if not path.exists():
        raise FileNotFoundError(f"No processed file for {dataset_id}. Run the {source} download script first.")
    frame = pd.read_csv(path, dtype={"geography": str, "series": str, "unit": str, "status": str},
                        na_values=[""], keep_default_na=False)
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    return frame


def series_of(frame, label_contains=""):
    """One series (values by date) whose label contains the given text."""
    labels = [s for s in frame["series"].unique() if label_contains.lower() in s.lower()]
    if len(labels) != 1:
        raise LookupError(f"expected one series matching {label_contains!r}, found {labels}")
    rows = frame[frame["series"] == labels[0]].dropna(subset=["date", "value"])
    return rows.set_index("date")["value"].sort_index()


POLICY_ID = "boc_v39079_policy_rate_daily"
CORRA_ID = "boc_avgintwo_corra_daily"
YIELDS_ID = "boc_bdcdnyld_bond_yields_daily"
FX_ID = "boc_fxusdcad_exchange_rate_daily"
datasets = {i: load_dataset(i) for i in [POLICY_ID, CORRA_ID, YIELDS_ID, FX_ID]}

policy = series_of(datasets[POLICY_ID])
corra = series_of(datasets[CORRA_ID])
y2 = series_of(datasets[YIELDS_ID], "2-year")
y10 = series_of(datasets[YIELDS_ID], "10-year")
usdcad = series_of(datasets[FX_ID])

In [ ]:
START = "2015-01-01"
p, c = policy.loc[START:], corra.loc[START:]

fig, ax = plt.subplots(figsize=(11, 4.4), constrained_layout=True)
ax.plot(c.index, c.values, color=ORANGE, lw=1.2, label="CORRA (overnight repo rate)")
ax.plot(p.index, p.values, color=BLUE, drawstyle="steps-post", label="Target for the overnight rate")
ax.set_ylim(bottom=0)
ax.margins(x=0.02)
ax.set_ylabel("Percent")
ax.set_title("Bank of Canada policy rate and CORRA", pad=24)
subtitle(ax, "Business daily. The policy rate is the Bank's target; CORRA is the rate actually paid on overnight repo funding.")
ax.legend(loc="upper left")
save_fig(fig, "boc_policy_rate_corra")
plt.show()

both = pd.concat({"policy": policy, "corra": corra}, axis=1).dropna()
recent = both.loc[START:]
spread_bp = (recent["corra"] - recent["policy"]) * 100
changes = policy.loc[START:].diff()
moves = changes[changes != 0].dropna()
display(Markdown("### Findings\n" + "\n".join([
    f"- **Now:** the policy rate is **{policy.iloc[-1]:.2f}%** and CORRA **{corra.iloc[-1]:.2f}%** ({corra.index[-1]:%b %d, %Y}).",
    f"- **Last change:** the target moved by {moves.iloc[-1] * 100:+.0f} basis points, effective {moves.index[-1]:%b %d, %Y}. "
    f"Since {START[:4]} it has changed {len(moves)} times, peaking at {policy.loc[START:].max():.2f}%.",
    f"- **CORRA tracks the target closely:** on {spread_bp.abs().le(5).mean():.0%} of days since {START[:4]} it was within "
    f"5 basis points of the target (median gap {spread_bp.median():+.1f} bp).",
    "- **Why it matters for risk:** the policy rate sets the floor for variable-rate mortgages and business loans, "
    "so its path drives borrowers' debt-service costs.",
])))

In [ ]:
curve = pd.concat({"y2": y2, "y10": y10}, axis=1).dropna()
spread = (curve["y10"] - curve["y2"]) * 100

fig, ax = plt.subplots(figsize=(11, 4.4), constrained_layout=True)
ax.axhline(0, color=AXIS, lw=1)
ax.fill_between(spread.index, spread.values, 0, where=spread.values < 0, interpolate=True,
                color=RED, alpha=0.25, lw=0)
ax.plot(spread.index, spread.values, color=BLUE, lw=1.2)
ax.margins(x=0.02)
ax.set_ylabel("Basis points")
ax.set_title("Yield curve slope: 10-year minus 2-year benchmark yield", pad=24)
subtitle(ax, "Government of Canada benchmark bonds, business daily. Shaded: inverted curve (2-year yield above 10-year).")
save_fig(fig, "boc_yield_curve_spread")
plt.show()

inverted = spread < 0
runs = (inverted != inverted.shift()).cumsum()[inverted]
episodes = [(g.index[0], g.index[-1]) for _, g in spread[inverted].groupby(runs) if len(g) >= 20]
display(Markdown("### Findings\n" + "\n".join([
    f"- **Now:** the 10-year yield is {y10.iloc[-1]:.2f}% and the 2-year {y2.iloc[-1]:.2f}%, a slope of "
    f"**{spread.iloc[-1]:+.0f} basis points** ({spread.index[-1]:%b %d, %Y}).",
    f"- **Inversions since {spread.index[0]:%Y}:** {len(episodes)} episodes of 20 or more trading days"
    + (", most recently " + "; ".join(f"{a:%b %Y} to {b:%b %Y}" for a, b in episodes[-3:]) if episodes else "") + ".",
    f"- **Deepest inversion:** {spread.min():+.0f} bp on {spread.idxmin():%b %d, %Y}.",
    "- **Why it matters for risk:** an inverted curve has historically preceded slowdowns, and it squeezes lenders "
    "that fund long loans with short-term money.",
    "- **Caveat:** these are yields on selected benchmark bonds, so a series can jump when the Bank moves the benchmark "
    "to a new bond (see `known_limitations` in the metadata record).",
])))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2), constrained_layout=True)
ax.plot(usdcad.index, usdcad.values, color=AQUA, lw=1.2)
ax.margins(x=0.06)
ax.set_ylabel("Canadian dollars per US dollar")
ax.set_title("USD/CAD exchange rate", pad=24)
subtitle(ax, "Bank of Canada daily average (indicative, for statistical and analytical use). Higher = weaker Canadian dollar.")
ax.annotate(f"{usdcad.iloc[-1]:.4f} ({usdcad.index[-1]:%b %Y})", (usdcad.index[-1], usdcad.iloc[-1]),
            xytext=(6, 0), textcoords="offset points", va="center", fontsize=9, color=INK2)
save_fig(fig, "boc_usd_cad")
plt.show()

year_ago = usdcad[usdcad.index <= usdcad.index[-1] - pd.DateOffset(years=1)].iloc[-1]
change = (usdcad.iloc[-1] / year_ago - 1) * 100
display(Markdown("### Findings\n" + "\n".join([
    f"- **Now:** one US dollar costs **{usdcad.iloc[-1]:.4f}** Canadian dollars ({usdcad.index[-1]:%b %d, %Y}), "
    + ("about the same as a year earlier." if abs(change) < 0.05 else
       f"{abs(change):.1f}% {'more' if change > 0 else 'less'} than a year earlier ({year_ago:.4f})."),
    f"- **Range since {usdcad.index[0]:%Y}:** from {usdcad.min():.4f} ({usdcad.idxmin():%b %Y}) to {usdcad.max():.4f} "
    f"({usdcad.idxmax():%b %Y}).",
    "- **Why it matters for risk:** a weaker Canadian dollar raises the cost of US-dollar debt and imports for "
    "Canadian businesses, and feeds into inflation.",
])))

In [ ]:
# Analysis-ready table 1: the four Bank of Canada datasets in one long table (standard columns, see the README).
daily = pd.concat(datasets.values(), ignore_index=True)
daily = daily[["date", "geography", "series", "value", "unit", "source", "dataset_id"]].sort_values(["dataset_id", "series", "date"])
daily["date"] = daily["date"].dt.strftime("%Y-%m-%d")
daily_path = ANALYSIS_READY / "policy_rate_corra_yields_usd_cad_daily.csv"
daily.to_csv(daily_path, index=False, encoding="utf-8")
print("Saved table:", daily_path.relative_to(HUB), f"({len(daily)} rows)")

# Analysis-ready table 2: monthly averages of the rates, with the unemployment rate from Statistics Canada.
# This is the first table in the hub that combines two publishers; every row keeps the dataset_id it came from.
unemployment = pd.read_csv(ANALYSIS_READY / "unemployment_rate_by_age_monthly.csv", parse_dates=["date"])
unemployment = unemployment[unemployment["age_group"] == "all_ages"]
parts = []
for dataset_id, series, label in [(POLICY_ID, policy, "Target for the overnight rate, monthly average"),
                                  (YIELDS_ID, y10, "Benchmark bond yield, 10-year, monthly average"),
                                  (FX_ID, usdcad, "USD/CAD, monthly average")]:
    averaged = series.resample("MS").mean().dropna()
    parts.append(pd.DataFrame({"date": averaged.index, "geography": "Canada", "series": label,
                               "value": averaged.round(4).values, "unit": datasets[dataset_id]["unit"].dropna().iloc[0],
                               "source": "boc", "dataset_id": dataset_id}))
parts.append(pd.DataFrame({"date": unemployment["date"], "geography": unemployment["geography"],
                           "series": "Unemployment rate, 15 years and over, seasonally adjusted",
                           "value": unemployment["value"], "unit": unemployment["unit"],
                           "source": unemployment["source"], "dataset_id": unemployment["dataset_id"]}))
monthly = pd.concat(parts, ignore_index=True)
# Keep only the months every series covers, so the table lines up for comparison.
coverage = monthly.groupby("date")["series"].nunique()
monthly = monthly[monthly["date"].isin(coverage[coverage == len(parts)].index)].sort_values(["series", "date"])
monthly["date"] = monthly["date"].dt.strftime("%Y-%m-%d")
monthly_path = ANALYSIS_READY / "rates_and_unemployment_monthly.csv"
monthly.to_csv(monthly_path, index=False, encoding="utf-8")
print("Saved table:", monthly_path.relative_to(HUB),
      f"({len(monthly)} rows, {monthly['date'].min()} to {monthly['date'].max()})")
print("Datasets combined:", sorted(monthly["dataset_id"].unique()))